# Mamba Cancer Gene Detector
**Task:** Given a raw DNA sequence → predict **Cancer (1)** or **Normal (0)**

---

| Cell | Name | What it does |
|------|------|--------------|
| 1 | Install | Install all required packages |
| 2 | GPU Check | Verify CUDA is available |
| 3 | Google Drive | Mount Drive for persistent storage |
| **4** | **PATHS** | **Set every input/output path ← EDIT HERE** |
| **5** | **DATA** | **Check / Upload / Download dataset ← KEY CELL** |
| 6 | Classes | Tokenizer, Dataset, Model definitions |
| 7 | Config | Training hyperparameters |
| 8 | Build | DataLoaders (80/10/10 split), Model, Optimizer |
| 9 | Train | Training loop with history tracking |
| 10 | Inference | Test the trained model on new sequences |
| **11** | **Evaluation** | **ROC/AUC, confusion matrix, learning curves** |
| **12** | **Baseline** | **Logistic Regression on k-mer features for comparison** |
| **13** | **Results** | **Final summary table: Mamba vs Baseline** |

---
**Recommended run order:** 1 → 2 → 3 → 4 → 5 → 6 → 7 → 8 → 9 → 10 → 11 → 12 → 13

---
## Cell 1 — Install Packages

> **Run once** at the start of every new Colab session.
> Set `REINSTALL = False` to skip installation if packages are already installed (saves ~2 minutes).

| Package | Purpose |
|---------|--------|
| `causal-conv1d` | CUDA kernel for 1-D causal convolution in Mamba blocks. Must be installed **before** `mamba-ssm`. |
| `mamba-ssm` | Core Mamba library — `MixerModel`, `MambaConfig`, selective-scan CUDA kernels. |
| `requests` | HTTP calls to GDC API (TCGA) and Ensembl REST API (DNA sequences). |
| `biopython` | DNA sequence utilities. |
| `pandas` | Reads/writes the dataset CSV. |
| `tqdm` | Progress bars during dataset download. |
| `transformers` | `get_cosine_schedule_with_warmup` for the LR schedule. |
| `scikit-learn` | ROC/AUC, confusion matrix, and logistic regression baseline (Cells 11–13). |
| `matplotlib` | Learning curve and ROC plots (Cell 11). |

| `REINSTALL` | Behaviour |
|-------------|-----------|
| `True` (default) | Runs all `pip install` commands — use at the start of a fresh session |
| `False` | Skips install — use if you already ran Cell 1 earlier in the same session |

> **Note:** `mamba-ssm` requires **Linux + CUDA**. It cannot compile on Windows — always use Google Colab.

In [ ]:
# -- Cell 1 -- Install Packages (skips packages already importable) --------
import importlib, subprocess, sys

def importable(name):
    try: importlib.import_module(name); return True
    except ImportError: return False

# Build prerequisites required for causal-conv1d / mamba-ssm CUDA compilation
if not (importable("causal_conv1d") and importable("mamba_ssm")):
    print("  Installing build prerequisites (packaging, ninja) ...", flush=True)
    subprocess.run("pip install packaging ninja -q", shell=True, check=True)

# (import_name, pip_install_spec, extra_flags, show_output)
# show_output=True  -> removes -q so compilation errors are visible
PKGS = [
    ("causal_conv1d", "causal-conv1d>=1.4.0", "--no-build-isolation", True),
    ("mamba_ssm",     "mamba-ssm",             "--no-build-isolation", True),
    ("Bio",           "biopython",              "",                     False),
    ("transformers",  "transformers",           "",                     False),
    ("sklearn",       "scikit-learn",           "",                     False),
    # requests, pandas, tqdm, matplotlib are pre-installed in Colab
]

all_ok = True
for imp, pip_spec, flags, show_output in PKGS:
    if importable(imp):
        print(f"  OK      {pip_spec}")
    else:
        print(f"  INSTALL {pip_spec} ...", flush=True)
        quiet = "" if show_output else "-q"
        cmd = f"pip install {pip_spec} {flags} {quiet}".strip()
        r = subprocess.run(cmd, shell=True)
        if r.returncode != 0:
            print(f"  FAILED  {pip_spec}")
            all_ok = False
        else:
            print(f"  DONE    {pip_spec}")

print("All packages ready." if all_ok else "WARNING: some installs failed.")


---
## Cell 2 — GPU Check

> **Before running:** go to **Runtime → Change runtime type → T4 GPU**.

### Why a GPU is required
`mamba-ssm` uses custom CUDA kernels — no CPU fallback exists.

### VRAM requirements by model size

| `d_model` | `n_layer` | Parameters | Min VRAM |
|-----------|-----------|------------|----------|
| 256 | 8 | ~10 M | ~1 GB |
| 512 | 12 | ~60 M | ~2.5 GB |
| 768 | 16 | ~130 M | ~5 GB |

The **Colab T4** has **15 GB VRAM** — sufficient for all sizes above.

In [4]:
import torch
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('=' * 45)
if torch.cuda.is_available():
    print(f'  GPU  : {torch.cuda.get_device_name(0)}')
    print(f'  VRAM : {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB')
    print(f'  CUDA : {torch.version.cuda}')
else:
    print('  WARNING: No GPU — go to Runtime → Change runtime type → GPU')
print('=' * 45)

ModuleNotFoundError: No module named 'torch'

---
## Cell 3 — Google Drive

> Mounts Drive so files persist across sessions. Colab resets all local storage after ~12 hours.

| Folder (default) | Contents |
|------------------|---------|
| `MyDrive/dataset/` | `cancer_genes.csv` — dataset |
| `MyDrive/dataset/cache/` | Raw `.maf.gz` files from TCGA |
| `MyDrive/dataset/checkpoints/` | `step_N.pt`, `final.pt` |

> Skip this cell only if using **Preset 2** (Colab local) in Cell 4.

In [ ]:
import os
try:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)
    DRIVE_AVAILABLE = True
    print('Google Drive mounted at /content/drive')
except ImportError:
    DRIVE_AVAILABLE = False
    print('Not in Colab — Drive not mounted. Local paths will be used.')

---
## Cell 3b — Clone Project from GitHub

> Clones the project repo into Google Drive so all files persist across sessions.
> **Run after Cell 3** (Drive must be mounted first).

| Variable | Default | Description |
|----------|---------|-------------|
| `REPO_URL` | your GitHub URL | HTTPS clone URL of the repo |
| `CLONE_DIR` | `/content/drive/MyDrive/Mamba-DNA-1` | Where to clone inside Drive |

**Behaviour:**
- Folder **does not exist** → clones automatically, no prompt.
- Folder **already exists** → asks *"Re-clone? (yes/no)"* before doing anything destructive.

In [ ]:
import os, shutil, subprocess, sys
from pathlib import Path

REPO_URL = 'https://github.com/mhmdnojim/mamba_brain_tumor_transformer_paper2_DNA.git'
DRIVE    = Path(DRIVE_DIR)
LOCAL    = Path(PIPELINE_DIR)

# ── Step 1: Sync CODE from GitHub -> Drive (data files never touched) ────────
print('=' * 65)
print('Step 1: Sync code  GitHub -> Drive  (data files are NEVER overwritten)')
print('=' * 65)

if not (DRIVE / '.git').exists():
    subprocess.run(['git', 'clone', REPO_URL, str(DRIVE)], check=True)
    print('  Cloned to Drive.')
else:
    subprocess.run(['git', '-C', str(DRIVE), 'fetch', 'origin'],
                   capture_output=True, check=True)
    # Only check out files that changed AND are code files (.py/.ipynb/.md)
    result = subprocess.run(
        ['git', '-C', str(DRIVE), 'diff', '--name-only', 'origin/main'],
        capture_output=True, text=True)
    changed = [f.strip() for f in result.stdout.splitlines()
               if f.strip().endswith(('.py', '.ipynb', '.md', '.txt'))]
    if changed:
        subprocess.run(
            ['git', '-C', str(DRIVE), 'checkout', 'origin/main', '--'] + changed,
            check=True)
        print(f'  Updated {len(changed)} code file(s): {changed}')
    else:
        print('  Code already up to date.')

# ── Step 2: Create local workspace dirs ──────────────────────────────────────
print()
print('Step 2: Create /content/work workspace')
print('=' * 65)
for d in ['sequences', 'variants', 'diagnostics/tmp_matching', 'dataset', 'checkpoints']:
    (LOCAL / d).mkdir(parents=True, exist_ok=True)

# Copy code files Drive -> LOCAL (skip if same size)
n_code = 0
for f in DRIVE.glob('*.py'):
    dst = LOCAL / f.name
    if not dst.exists() or f.stat().st_size != dst.stat().st_size:
        shutil.copy2(f, dst)
        n_code += 1
print(f'  {n_code} code file(s) updated in workspace.')

# ── Step 3: Mirror data files Drive -> LOCAL (skip if same size) ─────────────
print()
print('Step 3: Mirror data files  Drive -> /content/work  (skip if same size)')
print('=' * 65)

DATA_FILES = [
    'variants/TCGA-GBM_variants.tsv.gz',
    'variants/TCGA-BRCA_variants.tsv.gz',
    'sequences/TCGA-GBM_windows.jsonl.gz',
    'sequences/TCGA-BRCA_windows.jsonl.gz',
    'sequences/normals_windows.jsonl.gz',
    'sequences/normals_windows_matched.jsonl.gz',
    'dataset/cancer_genes_matched.csv',
    'dataset/cancer_genes.csv',
    'dataset/splits/train.csv',
    'dataset/splits/val.csv',
    'dataset/splits/test.csv',
    'diagnostics/gc_stats.json',
    'diagnostics/chrom_stats.json',
    'diagnostics/dinuc_stats.json',
    'diagnostics/summary.md',
    'diagnostics/tmp_matching/gencode.v38.annotation.gtf.gz',
    'diagnostics/tmp_matching/gencode.v38.annotation.bed',
]

for rel in DATA_FILES:
    src = DRIVE / rel
    dst = LOCAL / rel
    if not src.exists():
        print(f'  MISSING  {rel}  (will be created by pipeline)')
        continue
    src_sz = src.stat().st_size
    if dst.exists() and dst.stat().st_size == src_sz:
        print(f'  SKIP  {src_sz/1e6:7.1f} MB  {rel}')
    else:
        dst.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy2(src, dst)
        print(f'  COPY  {src_sz/1e6:7.1f} MB  {rel}')

# ── Step 4: GRCh38 FASTA (3.1 GB — symlink from Drive, never copy) ───────────
print()
print('Step 4: GRCh38 FASTA')
print('=' * 65)
fasta_drive = DRIVE / 'GRCh38_no_alt.fna'
fasta_local = LOCAL / 'GRCh38_no_alt.fna'
if fasta_local.exists() and not fasta_local.is_symlink():
    print(f'  OK      GRCh38_no_alt.fna  ({fasta_local.stat().st_size/1e9:.2f} GB, local copy)')
elif fasta_drive.exists():
    if fasta_local.is_symlink():
        fasta_local.unlink()   # refresh stale symlink
    os.symlink(str(fasta_drive), str(fasta_local))
    print(f'  LINKED  GRCh38_no_alt.fna  ({fasta_drive.stat().st_size/1e9:.2f} GB from Drive)')
    # Copy .fai index (small, needed for pyfaidx random access)
    fai_src = DRIVE / 'GRCh38_no_alt.fna.fai'
    fai_dst = LOCAL / 'GRCh38_no_alt.fna.fai'
    if fai_src.exists() and not fai_dst.exists():
        shutil.copy2(fai_src, fai_dst)
        print('  COPY    GRCh38_no_alt.fna.fai')
    elif fai_dst.exists():
        print('  SKIP    GRCh38_no_alt.fna.fai  (already present)')
    else:
        print('  NOTE    .fai index missing from Drive -- will be built on first run')
else:
    print('  MISSING: GRCh38_no_alt.fna not on Drive.')
    print('  Cell 5c (Path C) will download it automatically (~3.1 GB).')

# ── Step 5: Checkpoints ────────────────────────────────────────────────────
print()
print('Step 5: Checkpoints')
print('=' * 65)
ckpt_drive = DRIVE / 'checkpoints'
ckpt_local = LOCAL / 'checkpoints'
ckpt_local.mkdir(exist_ok=True)
if ckpt_drive.exists():
    pts = sorted(ckpt_drive.glob('*.pt'))
    for cp in pts:
        dst = ckpt_local / cp.name
        if dst.exists() and dst.stat().st_size == cp.stat().st_size:
            print(f'  SKIP  {cp.name}  ({cp.stat().st_size/1e6:.1f} MB)')
        else:
            shutil.copy2(cp, dst)
            print(f'  COPY  {cp.name}  ({cp.stat().st_size/1e6:.1f} MB)')
    if not pts:
        print('  No checkpoints on Drive yet.')
else:
    print('  No checkpoints on Drive yet.')

print()
print('Workspace ready.  PIPELINE_DIR =', str(LOCAL))


---
## Cell 4 — PATHS  ← **ONLY EDIT `PIPELINE_DIR`**

> Set `PIPELINE_DIR` to wherever Cell 3b cloned the repo. Everything else is derived automatically — no need to touch any other path.

### Folder layout inside the repo

```
Mamba-DNA-1/                  ← PIPELINE_DIR
├── 01_download_mafs.py
├── 02_extract_coords.py
├── 03_fetch_windows.py
├── 04_generate_negatives.py
├── 05_build_csv.py
├── mafs/                     ← Stage 1 output
├── variants/                 ← Stage 2 output
├── sequences/                ← Stage 3 & 4 output
└── dataset/                  ← created by Cell 4
    ├── cancer_genes.csv      ← DATA_CSV  (Stage 5 output, Cell 8 input)
    ├── cache/                ← CACHE_DIR  (legacy fallback)
    └── checkpoints/          ← CHECKPOINT_DIR  (model saves)
```

| Variable | Value (auto-derived) |
|----------|----------------------|
| `DATA_CSV` | `PIPELINE_DIR/dataset/cancer_genes.csv` |
| `CACHE_DIR` | `PIPELINE_DIR/dataset/cache` |
| `CHECKPOINT_DIR` | `PIPELINE_DIR/dataset/checkpoints` |

In [ ]:
import os
from pathlib import Path

# Drive = the canonical project (source of truth)
DRIVE_DIR    = '/content/drive/MyDrive/Mamba-DNA-1'
# /content/work = session-local workspace (fast disk, invisible to user)
PIPELINE_DIR = '/content/work'
DATA_CSV     = str(Path(PIPELINE_DIR) / 'dataset' / 'cancer_genes_matched.csv')

os.makedirs(PIPELINE_DIR, exist_ok=True)
print(f'DRIVE_DIR    = {DRIVE_DIR}')
print(f'PIPELINE_DIR = {PIPELINE_DIR}  (session workspace)')
print(f'DATA_CSV     = {DATA_CSV}')


---
## Cell 5 — DATA  (Key Cell)

> Ensures `DATA_CSV` contains a valid dataset before training. Three entry points depending on what you already have.

```
DATA_CSV exists?
  YES  → Option A: load it, show stats, done.
  NO   → UPLOAD_MY_OWN == True?
           YES → Option B: file picker opens.
           NO  → Option C: run the 5-stage pipeline to build the dataset.
```

### Option C — 5-Stage Pipeline (recommended for new sessions)

| Stage | Script | What it does | Time |
|-------|--------|--------------|------|
| 1 | `01_download_mafs.py` | GDC API → `.maf.gz` files under `PIPELINE_DIR/mafs/` | ~5 min |
| 2 | `02_extract_coords.py` | MAFs → `variants/*.tsv.gz` (chr, start, ref, alt per mutation) | ~1 min |
| 3 | `03_fetch_windows.py` | Ensembl API → `sequences/*_windows.jsonl.gz` — 512 bp DNA, **label=1** | ~1.7 h (GBM) |
| 4 | `04_generate_negatives.py` | Random genomic windows → `sequences/normals_windows.jsonl.gz` — **label=0** | ~same as stage 3 |
| 5 | `05_build_csv.py` | Merge all JSONL → `DATA_CSV` (shuffled, balanced) | <1 min |

> Stages 3 and 4 are **resumable** — restart after a crash and they skip work already done.

### User settings for Option C

| Variable | Default | What it controls |
|----------|---------|------------------|
| `UPLOAD_MY_OWN` | `False` | `True` → Colab file picker (Option B) |
| `RUN_STAGE` | `'all'` | `'all'` runs 1→5; or `'3'`, `'4'`, `'5'` to resume from a specific stage |
| `USE_DEMO` | `False` | `True` → synthetic data, no internet, no pipeline needed |

### Window specification
- Exactly **512 bp** centred on each variant's `Start_Position` (GRCh38)
- Anchor at index 255: `[start − 255, start + 256]`
- Telomere-edge variants dropped (not padded)
- All sequences on **+ strand**

### Expected CSV format
```
sequence,label
ATCGATCGATCG...,1
GCTAGCTAGCTA...,0
```

In [ ]:
import os, importlib.util, subprocess, sys
from pathlib import Path

# ── USER SETTINGS ─────────────────────────────────────────────────────────
UPLOAD_MY_OWN = False   # True → Colab file picker (Option B)
USE_DEMO      = False   # True → synthetic data, no internet needed
FORCE_RERUN   = False   # True → re-run a stage even if output already exists
# Which pipeline stages to run (Option C only):
#   'all' → run all 5 stages in order
#   '3'   → resume from stage 3 (MAFs + variants already done)
#   '5'   → only rebuild the CSV (all JSONL already done)
RUN_STAGE     = 'all'
# ──────────────────────────────────────────────────────────────────────────

def _run_stage(script_name):
    script = os.path.join(PIPELINE_DIR, script_name)
    if not os.path.exists(script):
        raise FileNotFoundError(f'{script} not found.')
    print(f'\n{chr(9472)*55}')
    print(f'  Running {script_name} ...')
    print(f'{chr(9472)*55}', flush=True)
    subprocess.run(
        [sys.executable, script],
        cwd=PIPELINE_DIR, check=True,
        env={**os.environ, 'PYTHONUNBUFFERED': '1'},
    )

def _stage_complete(stage):
    p = Path(PIPELINE_DIR)
    if stage == '1':
        gbm  = list((p/'mafs'/'TCGA-GBM').glob('*.maf.gz'))  if (p/'mafs'/'TCGA-GBM').exists()  else []
        brca = list((p/'mafs'/'TCGA-BRCA').glob('*.maf.gz')) if (p/'mafs'/'TCGA-BRCA').exists() else []
        if gbm and brca:
            return True, f'{len(gbm)} GBM + {len(brca)} BRCA .maf.gz files present'
    elif stage == '2':
        gbm  = p / 'variants' / 'TCGA-GBM_variants.tsv.gz'
        brca = p / 'variants' / 'TCGA-BRCA_variants.tsv.gz'
        if gbm.exists() and brca.exists():
            mb = (gbm.stat().st_size + brca.stat().st_size) / 1e6
            return True, f'variant TSVs exist ({mb:.1f} MB)'
    elif stage == '3':
        gbm  = p / 'sequences' / 'TCGA-GBM_windows.jsonl.gz'
        brca = p / 'sequences' / 'TCGA-BRCA_windows.jsonl.gz'
        files = [f for f in [gbm, brca] if f.exists()]
        if files:
            mb = sum(f.stat().st_size for f in files) / 1e6
            return True, f'{len(files)}/2 window file(s) exist ({mb:.1f} MB)'
    elif stage == '4':
        matched = p / 'sequences' / 'normals_windows_matched.jsonl.gz'
        normals = p / 'sequences' / 'normals_windows.jsonl.gz'
        if matched.exists() and matched.stat().st_size > 0:
            mb = matched.stat().st_size / 1e6
            return True, f'matched negatives exist ({mb:.1f} MB)'
        if normals.exists() and normals.stat().st_size > 0:
            mb = normals.stat().st_size / 1e6
            return True, f'normals file exists ({mb:.1f} MB)'
    elif stage == '5':
        csv = Path(DATA_CSV)
        if csv.exists():
            mb = csv.stat().st_size / 1e6
            return True, f'cancer_genes.csv exists ({mb:.1f} MB)'
    return False, ''

def _show_csv_stats():
    import pandas as pd
    df = pd.read_csv(DATA_CSV)
    print('=' * 55)
    print(f'  Path    : {DATA_CSV}')
    print(f'  Rows    : {len(df):,}')
    print(f'  Cancer  : {(df["label"]==1).sum():,}')
    print(f'  Normal  : {(df["label"]==0).sum():,}')
    balance = (df['label']==0).sum() / max(1, len(df)) * 100
    print(f'  Balance : {balance:.1f}% normal')
    print('=' * 55)

if os.path.exists(DATA_CSV) and not UPLOAD_MY_OWN:
    print('OPTION A - Dataset already exists')
    _show_csv_stats()

elif UPLOAD_MY_OWN:
    print('OPTION B - Upload your CSV')
    try:
        from google.colab import files
        import shutil
        uploaded = files.upload()
        shutil.copy(list(uploaded.keys())[0], DATA_CSV)
        _show_csv_stats()
    except ImportError:
        print(f'  Not in Colab. Copy your CSV manually to: {DATA_CSV}')

else:
    if USE_DEMO:
        print('DEMO - Generating synthetic dataset...')
        spec = importlib.util.spec_from_file_location(
            'dataset', os.path.join(PIPELINE_DIR, 'dataset.py'))
        if spec:
            ds = importlib.util.module_from_spec(spec)
            spec.loader.exec_module(ds)
            ds.generate_demo_dataset(out_path=DATA_CSV, n_samples=400, seq_len=512)
        if os.path.exists(DATA_CSV):
            _show_csv_stats()
    else:
        print('OPTION C - Running 5-stage TCGA pipeline')
        print(f'  Pipeline root : {PIPELINE_DIR}')
        print(f'  Output CSV    : {DATA_CSV}')
        print(f'  Starting from : stage {RUN_STAGE}')
        print(f'  Force rerun   : {FORCE_RERUN}')

        STAGE_ORDER = ['1', '2', '3', '4', '5']
        start_idx   = 0 if RUN_STAGE == 'all' else STAGE_ORDER.index(RUN_STAGE)
        scripts = {
            '1': '01_download_mafs.py',
            '2': '02_extract_coords.py',
            '3': '03_fetch_windows.py',
            '5': '05_build_csv.py',
        }
        # Stage 3 self-resumes -- always run to pick up missing records.
        # Stage 4 = matched negatives -- handled by Cells 5b/5c/5d.
        ALWAYS_RUN = {'3'}

        for stage in STAGE_ORDER[start_idx:]:
            done, reason = _stage_complete(stage)

            if stage == '4':
                if done:
                    print(f'\n  Stage 4 -- skipped (done): {reason}')
                else:
                    print(f'\n  Stage 4 -- SKIPPED')
                    print(f'  Run Cell 5b then Cell 5c for matched negatives.')
                continue

            if done:
                if stage in ALWAYS_RUN:
                    print(f'\n  Stage {stage} -- {reason}')
                    print(f'           Resuming ...')
                elif not FORCE_RERUN:
                    print(f'\n  Stage {stage} -- skipped (done): {reason}')
                    continue
                else:
                    print(f'\n  Stage {stage} -- forcing re-run')
            else:
                print(f'\n  Stage {stage} -- starting ...')

            if stage == '5':
                script = os.path.join(PIPELINE_DIR, scripts['5'])
                print(f'{chr(9472)*55}')
                print(f'  Running {scripts["5"]} ...')
                print(f'{chr(9472)*55}', flush=True)
                subprocess.run(
                    [sys.executable, script, '--out', DATA_CSV],
                    cwd=PIPELINE_DIR, check=True,
                    env={**os.environ, 'PYTHONUNBUFFERED': '1'})
            else:
                _run_stage(scripts[stage])

        if os.path.exists(DATA_CSV):
            _show_csv_stats()
        else:
            print('\n  Stage 3 complete.')
            print('  Next: run Cell 5b -> 5c -> 5d for matched negatives and CSV.')


---
## Cell 5b — Diagnostics & Matching Decision

> Runs `06_diagnostics.py --no_ensembl` (~1 minute, no network calls), then reads the JSON outputs
> and automatically decides which biases are severe enough to require matched negatives.
>
> **Run this before any matching step.** The output tells you exactly what to fix and which
> pipeline path (04b subsampling vs 04c full matching) is warranted.

| Diagnostic | Test used | Threshold for action |
|------------|-----------|---------------------|
| GC content | KS statistic + Δmean | HIGH if \|Δmean\| > 0.05 |
| Chromosome | χ² p-value | HIGH if p < 1e-10 |
| Dinucleotide | Welch t-test per dinuc | Informational only |

**Decision outputs:**
- Which variables need matching (GC / chromosome / region type)
- Whether bedtools or REST is required
- Exact next cell to run

In [ ]:
# ── Cell 5b — Run Diagnostics & Get Matching Decision ────────────────────────
import subprocess, sys, json, os
from pathlib import Path

DIAG_DIR = Path(PIPELINE_DIR) / 'diagnostics'

# ── Step 1: Run fast diagnostics (no Ensembl, ~1 minute) ─────────────────────
print('Running 06_diagnostics.py --no_ensembl  (~1 minute, no network) ...')
print('─' * 65, flush=True)
result = subprocess.run(
    [sys.executable, '06_diagnostics.py', '--no_ensembl'],
    cwd=PIPELINE_DIR,
    capture_output=False,
    text=True,
)
if result.returncode != 0:
    print(f'\nWARNING: diagnostics script exited with code {result.returncode}')
    print('Make sure 06_diagnostics.py is present in PIPELINE_DIR and the')
    print('sequences/ folder exists (run Cell 5 first).')

# ── Step 2: Load JSON results ─────────────────────────────────────────────────
def _load_json(name):
    p = DIAG_DIR / name
    if p.exists():
        return json.loads(p.read_text())
    return None

gc_stats    = _load_json('gc_stats.json')
chrom_stats = _load_json('chrom_stats.json')
dinuc_stats = _load_json('dinuc_stats.json')

missing = [n for n, d in [('gc_stats.json', gc_stats),
                           ('chrom_stats.json', chrom_stats),
                           ('dinuc_stats.json', dinuc_stats)] if d is None]
if missing:
    print(f'\nERROR: Missing output file(s): {missing}')
    print('Check that 06_diagnostics.py ran successfully above.')
else:
    # ── Step 3: Parse severity & build decision ───────────────────────────────
    delta_gc  = abs(gc_stats.get('delta_mean', 0.0))
    ks_stat   = gc_stats.get('ks_statistic', 0.0)
    ks_pval   = gc_stats.get('ks_pvalue', 1.0)
    chi2_p    = chrom_stats.get('chi2_pvalue', 1.0)

    def sev_gc(d):
        return 'HIGH' if d > 0.05 else ('MEDIUM' if d > 0.02 else 'LOW')

    def sev_p(p):
        return 'HIGH' if p < 1e-10 else ('MEDIUM' if p < 1e-3 else 'LOW')

    gc_sev    = sev_gc(delta_gc)
    chrom_sev = sev_p(chi2_p)

    # Dinucleotide: flag CpG and any dinuc with |delta| > 0.01 and p < 1e-5
    DINUCS = [a + b for a in 'ACGT' for b in 'ACGT']
    dinuc_flags = []
    if dinuc_stats and 'delta' in dinuc_stats and 'pvalue' in dinuc_stats:
        for i, dn in enumerate(DINUCS):
            d = dinuc_stats['delta'][i]
            p = dinuc_stats['pvalue'][i]
            if abs(d) > 0.01 and p < 1e-5:
                dinuc_flags.append((dn, d, p))
        dinuc_flags.sort(key=lambda x: abs(x[1]), reverse=True)

    # ── Print decision report ─────────────────────────────────────────────────
    print('\n' + '=' * 65)
    print('  DIAGNOSTIC DECISION REPORT  (BEFORE matching)')
    print('=' * 65)
    print(f"  GC content    delta={gc_stats.get('delta_mean', 0):+.4f}  "
          f"KS={ks_stat:.3f}  KS-p={ks_pval:.2e}  ->  {gc_sev}")
    print(f"  Chromosome    chi2_p={chi2_p:.2e}                    ->  {chrom_sev}")
    if dinuc_flags:
        top5 = dinuc_flags[:5]
        flag_str = '  '.join(f"{dn}({d:+.4f})" for dn, d, p in top5)
        print(f"  Dinucleotide  top biased: {flag_str}")
    else:
        print(f"  Dinucleotide  no strong per-dinuc bias detected")
    print('-' * 65)

    # Decision flags
    match_gc    = delta_gc > 0.02
    match_chrom = chi2_p < 1e-3
    need_region = delta_gc > 0.05

    print('\n  MATCHING DECISION:')
    print(f"  Match GC content   : {'YES  (delta={:.4f})'.format(delta_gc) if match_gc else 'NO   (delta={:.4f})'.format(delta_gc)}")
    print(f"  Match chromosome   : {'YES  (p={:.2e})'.format(chi2_p) if match_chrom else 'NO   (p={:.2e})'.format(chi2_p)}")
    print(f"  Check region type  : {'YES -- use Path C in Cell 5c' if need_region else 'OPTIONAL (GC bias is moderate)'}")

    print('\n  PIPELINE RECOMMENDATION:')
    if not match_gc and not match_chrom:
        print('  [ LOW bias ]  Proceed directly to Cell 6 (training).')
    elif match_gc and not need_region:
        print('  [ MEDIUM bias ]  Run Cell 5c with MATCHING_PATH = "B" (GC+chr, ~10 min).')
    else:
        print('  [ HIGH bias ]  Run Cell 5c with MATCHING_PATH = "C" (recommended, ~15 min).')

    print('\n  WHAT DRIVES THE GC BIAS:')
    print('  Cancer mutations cluster in GC-rich open chromatin (promoters, enhancers).')
    print('  Random genome-wide negatives sample GC-poor regions (genome avg ~39%).')
    if delta_gc > 0.05:
        print(f'  delta={gc_stats.get("delta_mean", 0):+.4f} is LARGE -- model learns GC, not biology.')

    print('=' * 65)

    # ── Step 4: Print summary.md ──────────────────────────────────────────────
    summary_path = DIAG_DIR / 'summary.md'
    if summary_path.exists():
        print('\n--- diagnostics/summary.md ---')
        print(summary_path.read_text())
    else:
        print('\n(summary.md not written -- check 06_diagnostics.py completed)')

    # ── Store baseline for Cell 5e before/after comparison ───────────────────
    BEFORE_GC_DELTA  = gc_stats.get('delta_mean', 0.0)
    BEFORE_GC_KS     = gc_stats.get('ks_statistic', 0.0)
    BEFORE_GC_KS_P   = gc_stats.get('ks_pvalue', 1.0)
    BEFORE_CHR_CHI2P = chrom_stats.get('chi2_pvalue', 1.0)
    print(f'\n  Baseline stored: GC_delta={BEFORE_GC_DELTA:+.4f}  '
          f'KS={BEFORE_GC_KS:.3f}  chr_p={BEFORE_CHR_CHI2P:.2e}'
          f'  (Cell 5e will compare against these)')

---
## Cell 5c — Generate Matched Negatives

> Installs bedtools, downloads reference files (if needed), then runs `04c_generate_negatives_matched.py`.
> **Run only if Cell 5b reported MEDIUM or HIGH bias.**

### Choose your path

| `MATCHING_PATH` | What it matches | Time | Downloads |
|---|---|---|---|
| `'A'` | chromosome + GC (Ensembl for sequences) | ~5-8 h | nothing |
| `'B'` | chromosome + GC (local FASTA, fast) | ~10 min | GRCh38.fna (1.1 GB) |
| `'C'` | chromosome + GC + region type (recommended) | ~15 min | GRCh38.fna + GENCODE GTF (auto) |

Set `MATCHING_PATH = 'C'` if Cell 5b said **HIGH bias** (likely: GC Δ > 0.05).

> Output: `sequences/normals_windows_matched.jsonl.gz` — one matched negative per positive.

In [ ]:
# ── Cell 5c — Generate Matched Negatives ─────────────────────────────────────
import os, subprocess, sys, gzip
from pathlib import Path

# ── USER SETTINGS ─────────────────────────────────────────────────────────────
MATCHING_PATH = 'B'              # 'A' = Ensembl (~8h), 'B' = local FASTA (~10min),
                                 # 'C' = FASTA + GENCODE region type (21% coverage with TCGA WXS -- avoid)
# MATCHING_PATH = 'C'           # previous setting — region-type matching (use for WGS data)
REF_FASTA     = 'GRCh38_no_alt.fna'  # filename inside PIPELINE_DIR (Path B/C only)
GC_TOL        = 0.03            # +/- GC tolerance (0.03 = +-3%, triples match rate vs 0.02)
OVERSAMPLE    = 50              # bedtools candidate oversample factor
# OVERSAMPLE  = 100             # previous setting — needed for Path C exonic matching
MAX_TRIES     = 500             # max tries per positive before skipping (04c default was 300)
FORCE_REGEN   = False           # True -> regenerate even if output already exists
# ─────────────────────────────────────────────────────────────────────────────

os.chdir(PIPELINE_DIR)
ref_path = Path(PIPELINE_DIR) / REF_FASTA
out_file = Path(PIPELINE_DIR) / 'sequences' / 'normals_windows_matched.jsonl.gz'

# ── Step 1: Install bedtools ──────────────────────────────────────────────────
print('[1/4] Checking bedtools ...')
try:
    r = subprocess.run(['bedtools', '--version'], capture_output=True, text=True,
                       check=True)
    print(f'  {r.stdout.strip()}')
except (FileNotFoundError, subprocess.CalledProcessError):
    print('  bedtools not found. Installing ...')
    subprocess.run(['apt-get', 'install', '-y', '-q', 'bedtools'], check=True)
    print('  bedtools installed.')

# ── Step 2: Download GRCh38 FASTA (Path B/C only) ────────────────────────────
if MATCHING_PATH in ('B', 'C'):
    if ref_path.exists():
        mb = ref_path.stat().st_size / 1e6
        print(f'\n[2/4] GRCh38 FASTA already present ({mb:.0f} MB) -- skip download.')
    else:
        print('\n[2/4] Downloading GRCh38 no-alt FASTA (~1.1 GB, ~5 min) ...')
        fna_gz = Path(PIPELINE_DIR) / 'GRCh38_no_alt.fna.gz'
        url = ('https://ftp.ncbi.nlm.nih.gov/genomes/all/GCA/000/001/405/'
               'GCA_000001405.15_GRCh38/seqs_for_alignment_pipelines.ucsc_ids/'
               'GCA_000001405.15_GRCh38_no_alt_analysis_set.fna.gz')
        subprocess.run(['wget', '-q', '--show-progress', '-O', str(fna_gz), url],
                       check=True)
        print('  Decompressing ...')
        subprocess.run(['gunzip', '-f', str(fna_gz)], check=True)
        print(f'  Done. {ref_path}')
else:
    print('\n[2/4] Path A: Ensembl REST -- no FASTA download needed.')

# ── Step 3: Build 04c command ─────────────────────────────────────────────────
print(f'\n[3/4] Preparing 04c command (Path {MATCHING_PATH}) ...')
cmd = [sys.executable, '04c_generate_negatives_matched.py',
       '--gc_tol', str(GC_TOL),
       '--oversample', str(OVERSAMPLE),
       '--max_tries', str(MAX_TRIES)]

if MATCHING_PATH in ('B', 'C'):
    cmd += ['--ref', str(ref_path)]
if MATCHING_PATH == 'B':
    cmd += ['--no_region']         # GC + chromosome only (no region type)
# Path C: --ref only; 04c auto-downloads GENCODE GTF if not present
# Path A: no extra flags; Ensembl used for everything

print(f'  {" ".join(str(c) for c in cmd)}')

# ── Step 4: Run 04c (auto-resumes from partial output) ───────────────────────
if out_file.exists() and out_file.stat().st_size == 0:
    print("[4/4] Empty output file (crashed run) -- deleting.")
    out_file.unlink()

if FORCE_REGEN and out_file.exists():
    print("[4/4] FORCE_REGEN=True -- deleting existing output.")
    out_file.unlink()

print(f"
[4/4] Running 04c ... (Path {MATCHING_PATH}, auto-resumes if partial)")
proc = subprocess.Popen(
    cmd, cwd=PIPELINE_DIR, stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT, text=True,
    env={**os.environ, "PYTHONUNBUFFERED": "1"})
for raw in proc.stdout:
    print(raw, end="", flush=True)
proc.wait()
if proc.returncode != 0:
    raise subprocess.CalledProcessError(proc.returncode, cmd)
if out_file.exists():
    mb = out_file.stat().st_size / 1e6
    n  = sum(1 for _ in gzip.open(out_file, 'rt'))
    print(f'\n  Output: sequences/normals_windows_matched.jsonl.gz')
    print(f'          {n:,} matched negatives  ({mb:.1f} MB)')
    print('\n  Next: run Cell 5d to rebuild the training CSV.')

---
## Cell 5d — Rebuild Training CSV

> Merges cancer windows + matched negatives into a new `cancer_genes_matched.csv`.
> This replaces the original unmatched CSV for training.

After this cell completes, update `DATA_CSV` in **Cell 4** to point at the matched CSV,
then re-run **Cells 8 → 9** to train on the clean dataset.

In [ ]:
# ── Cell 5d — Rebuild Training CSV with Matched Negatives ────────────────────
import subprocess, sys, os
from pathlib import Path

MATCHED_NEG = Path(PIPELINE_DIR) / 'sequences' / 'normals_windows_matched.jsonl.gz'
MATCHED_CSV = Path(PIPELINE_DIR) / 'dataset'   / 'cancer_genes_matched.csv'

if not MATCHED_NEG.exists():
    print('ERROR: normals_windows_matched.jsonl.gz not found.')
    print('Run Cell 5c first to generate matched negatives.')
else:
    print('Building matched dataset CSV ...')
    script = os.path.join(PIPELINE_DIR, '05_build_csv.py')
    subprocess.run(
        [sys.executable, script,
         '--negatives', str(MATCHED_NEG),
         '--out', str(MATCHED_CSV)],
        cwd=PIPELINE_DIR, check=True,
        env={**os.environ, 'PYTHONUNBUFFERED': '1'}
    )

    if MATCHED_CSV.exists():
        import pandas as pd
        df = pd.read_csv(MATCHED_CSV)
        balance = (df['label'] == 0).sum() / max(1, len(df)) * 100
        print('\n' + '=' * 55)
        print(f'  Matched CSV   : {MATCHED_CSV}')
        print(f'  Total rows    : {len(df):,}')
        print(f'  Cancer (1)    : {(df["label"]==1).sum():,}')
        print(f'  Normal (0)    : {(df["label"]==0).sum():,}')
        print(f'  Balance       : {balance:.1f}% normal  '
              f'{"✓ balanced" if 40 < balance < 60 else "⚠ imbalanced"}')
        print('=' * 55)
        print('\n  ACTION REQUIRED: update DATA_CSV in Cell 4 then re-run Cells 8->9:')
        print(f'  DATA_CSV = "{MATCHED_CSV}"')
        # Also update the module-level variable for this session
        DATA_CSV = str(MATCHED_CSV)
        print(f'\n  DATA_CSV updated in this session to: {DATA_CSV}')

        # Chromosome distribution (quick sanity check)
        chrom_col = 'chromosome' if 'chromosome' in df.columns else 'chrom'
        if chrom_col in df.columns:
            print('\n  Chromosome distribution (top 5):')
            top = df[chrom_col].value_counts().head(5)
            for ch, cnt in top.items():
                print(f'    {ch:<8} {cnt:,}')
        print('  Next: run Cell 5e to verify bias is eliminated.')

---
## Cell 5e — Diagnostics AFTER Matching (Proof it Worked)

> Re-runs `06_diagnostics.py --no_ensembl` on the **matched** negatives and compares
> against the baseline numbers stored in Cell 5b.
>
> This produces the before/after table that goes in the paper's Methods section.

**Expected result after good matching:**

| Metric | Before | After (target) |
|---|---|---|
| GC Δ mean | +0.09 | < 0.005 |
| GC KS p-value | < 1e-50 | > 0.05 |
| Chr χ² p-value | < 1e-10 | > 0.05 |

If all three pass, copy the `"After matched-negative sampling..."` sentence into the paper.

In [ ]:
# ── Cell 5e — Diagnostics AFTER Matching (before/after comparison) ────────────
import subprocess, sys, json
from pathlib import Path

DIAG_DIR    = Path(PIPELINE_DIR) / 'diagnostics'
MATCHED_NEG = Path(PIPELINE_DIR) / 'sequences' / 'normals_windows_matched.jsonl.gz'

if not MATCHED_NEG.exists():
    print('ERROR: normals_windows_matched.jsonl.gz not found. Run Cell 5c first.')
else:
    # ── Step 1: Run diagnostics on the matched negatives ──────────────────────
    print('Running 06_diagnostics.py --no_ensembl on MATCHED negatives (~1 min) ...')
    print('─' * 65, flush=True)
    subprocess.run(
        [sys.executable, '06_diagnostics.py',
         '--no_ensembl',
         '--normals', str(MATCHED_NEG)],
        cwd=PIPELINE_DIR, check=True,
        env={**os.environ, 'PYTHONUNBUFFERED': '1'}
    )

    # ── Step 2: Load after-matching results ───────────────────────────────────
    def _load(name):
        p = DIAG_DIR / name
        return json.loads(p.read_text()) if p.exists() else None

    gc_after    = _load('gc_stats.json')
    chrom_after = _load('chrom_stats.json')

    if not gc_after or not chrom_after:
        print('\nERROR: JSON files not found -- check diagnostics run above.')
    else:
        # ── Step 3: Before/after comparison table ─────────────────────────────
        # Retrieve before-matching values stored by Cell 5b
        # (defaults used if Cell 5b was not run in this session)
        b_gc_d  = globals().get('BEFORE_GC_DELTA',  None)
        b_gc_ks = globals().get('BEFORE_GC_KS',     None)
        b_gc_p  = globals().get('BEFORE_GC_KS_P',   None)
        b_chr_p = globals().get('BEFORE_CHR_CHI2P', None)

        a_gc_d  = gc_after['delta_mean']
        a_gc_ks = gc_after['ks_statistic']
        a_gc_p  = gc_after['ks_pvalue']
        a_chr_p = chrom_after['chi2_pvalue']

        def fmt(v, fmt_str='{:.4f}'):
            return fmt_str.format(v) if v is not None else '(run Cell 5b)'

        print('\n' + '=' * 65)
        print('  BEFORE vs AFTER MATCHING')
        print('=' * 65)
        print(f"  {'Metric':<24} {'BEFORE':>12}  {'AFTER':>12}  {'Change':>10}")
        print('─' * 65)
        print(f"  {'GC delta_mean':<24} {fmt(b_gc_d, '{:+.4f}'):>12}  "
              f"{a_gc_d:>+12.4f}  "
              f"{'improved' if b_gc_d is not None and abs(a_gc_d) < abs(b_gc_d) else '':>10}")
        print(f"  {'GC KS statistic':<24} {fmt(b_gc_ks):>12}  "
              f"{a_gc_ks:>12.4f}")
        print(f"  {'GC KS p-value':<24} {fmt(b_gc_p, '{:.2e}'):>12}  "
              f"{a_gc_p:>12.2e}")
        print(f"  {'Chrom chi2 p-value':<24} {fmt(b_chr_p, '{:.2e}'):>12}  "
              f"{a_chr_p:>12.2e}")
        print('=' * 65)

        # ── Step 4: Verdict ───────────────────────────────────────────────────
        gc_ok  = abs(a_gc_d) < 0.005 and a_gc_p > 0.05
        chr_ok = a_chr_p > 0.01

        print('\n  VERDICT:')
        print(f"  GC bias   : {'PASS (eliminated)' if gc_ok else 'FAIL -- still biased'}")
        print(f"  Chr bias  : {'PASS (eliminated)' if chr_ok else 'FAIL -- still biased'}")

        if gc_ok and chr_ok:
            print('\n  ALL PASS -- matching worked. Safe to train and submit.')
        elif not gc_ok:
            print('\n  GC bias remains. Try: lower --gc_tol or increase --oversample in Cell 5c.')
        else:
            print('\n  Chromosome bias remains. Make sure bedtools -chrom flag is active.')

        # ── Step 5: Paper methods text ────────────────────────────────────────
        print('\n  COPY THIS INTO YOUR PAPER (Methods section):')
        print('  ─' * 32)
        print(f'  "After matched-negative sampling (chromosome-matched, GC-matched')
        print(f'   to within +-{0.02:.0%}, region-type-matched using GENCODE v38),')
        print(f'   GC content was indistinguishable between classes')
        print(f'   (KS p = {a_gc_p:.2f}, delta_mean = {a_gc_d:+.4f},')
        print(f'   chromosome chi2 p = {a_chr_p:.2f})."')
        print('  ─' * 32)

        # ── Step 6: Print updated summary.md ─────────────────────────────────
        summary_path = DIAG_DIR / 'summary.md'
        if summary_path.exists():
            print('\n--- diagnostics/summary.md (AFTER matching) ---')
            print(summary_path.read_text())
        # ── Step 7: Archive diagnostics to Drive ─────────────────────────────
        import shutil
        _dst = Path(DRIVE_DIR) / 'reports' / 'diagnostics_after_matching'
        _dst.mkdir(parents=True, exist_ok=True)
        _n = 0
        for _f in DIAG_DIR.glob('*'):
            if _f.is_file():
                shutil.copy2(_f, _dst / _f.name)
                _n += 1
        _figs_src = DIAG_DIR / 'figs'
        if _figs_src.exists():
            (_dst / 'figs').mkdir(exist_ok=True)
            for _f in _figs_src.glob('*'):
                if _f.is_file():
                    shutil.copy2(_f, _dst / 'figs' / _f.name)
                    _n += 1
        print(f'\n  Archived {_n} diagnostic file(s) -> {_dst}')
        print('  Next: run Cell 5d to rebuild the training CSV.')

---
## Cell 5f — Chromosome-Level Train / Val / Test Split

> **Why:** a random 80/10/10 split leaks data — two variants 200 bp apart on the same
> chromosome land in both train and test, inflating AUC. Held-out chromosomes eliminate this.

| Split | Chromosomes held out | Reason |
|-------|----------------------|--------|
| Test  | chr20, chr22 | Both ~50% cancer balance; chr19 excluded (62% imbalance due to high gene density) |
| Val   | chr8, chr21  | Moderate size; chr21 (Down syndrome) biologically distinct |
| Train | everything else | ~80% of data |

**Requires:** Cell 5d to have been run with the fixed `05_build_csv.py`
(CSV must have `chromosome` column). Pull latest code if you see a `KeyError`.

Output: `dataset/splits/{train,val,test}.csv`


In [ ]:
# ── Cell 5f — Chromosome-Level Split ────────────────────────────────────────
import pandas as pd
from pathlib import Path

CSV_IN  = Path(PIPELINE_DIR) / 'dataset' / 'cancer_genes_matched.csv'
OUT_DIR = Path(PIPELINE_DIR) / 'dataset' / 'splits'
OUT_DIR.mkdir(parents=True, exist_ok=True)

if not CSV_IN.exists():
    print('ERROR: cancer_genes_matched.csv not found. Run Cell 5d first.')
elif 'chromosome' not in pd.read_csv(CSV_IN, nrows=1).columns:
    print('ERROR: CSV is missing the chromosome column.')
    print('Pull the latest 05_build_csv.py fix and re-run Cell 5d.')
else:
    df = pd.read_csv(CSV_IN)
    print(f'Loaded {len(df):,} rows')

    TEST_CHROMS = {'chr19', 'chr22'}
    VAL_CHROMS  = {'chr8',  'chr21'}

    test_df  = df[df['chromosome'].isin(TEST_CHROMS)].reset_index(drop=True)
    val_df   = df[df['chromosome'].isin(VAL_CHROMS)].reset_index(drop=True)
    train_df = df[~df['chromosome'].isin(TEST_CHROMS | VAL_CHROMS)].reset_index(drop=True)

    # Sanity checks — hard fail if leakage detected
    assert len(train_df) + len(val_df) + len(test_df) == len(df), 'row count mismatch'
    assert set(train_df['chromosome']).isdisjoint(TEST_CHROMS | VAL_CHROMS), 'train/test leakage'
    assert set(val_df['chromosome']).isdisjoint(TEST_CHROMS), 'val/test overlap'

    def summarize(name, d):
        n, pos = len(d), int((d['label'] == 1).sum())
        print(f'  {name:5s}: n={n:>7,}  cancer={pos:>7,}  normal={n-pos:>7,}  '
              f'cancer_frac={pos/n:.3f}  ({n/len(df):.1%} of total)')

    print('\n=== Split sizes ===')
    summarize('train', train_df)
    summarize('val',   val_df)
    summarize('test',  test_df)

    # Per-chromosome class balance — flags poor matching coverage
    print('\n=== Class balance per chromosome (cancer fraction; ideal ~0.50) ===')
    bal = df.groupby('chromosome')['label'].agg(['count', 'sum'])
    bal['cancer_frac'] = bal['sum'] / bal['count']
    bal = bal.sort_values('cancer_frac', ascending=False)
    print(bal.to_string())

    worst = bal[bal['cancer_frac'] > 0.55].index.tolist()
    if worst:
        print(f'\n  WARNING: chromosomes with >55% cancer (poor coverage): {worst}')
    else:
        print('\n  All chromosomes within 45-55% cancer -- matching coverage uniform.')

    # Save splits
    keep_cols = ['sequence', 'label', 'chromosome']
    for name, d in [('train', train_df), ('val', val_df), ('test', test_df)]:
        out = OUT_DIR / f'{name}.csv'
        d[keep_cols].to_csv(out, index=False)
        print(f'  Wrote: {out.name}  ({out.stat().st_size/1e6:.1f} MB)')

    # Update session variable so Cell 8 picks up correct paths automatically
    SPLIT_DIR = str(OUT_DIR)
    print(f'\n  SPLIT_DIR set to: {SPLIT_DIR}')
    print('  Next: run Cell 8 (it will load from dataset/splits/ automatically).')


---
## Cell 6 — Classes

> All model, dataset, and helper definitions. Nothing here needs to be changed.

### `DNATokenizer`
Character-level tokenizer. Vocabulary: `PAD=0  UNK=1  A=2  T=3  G=4  C=5  N=6`  
Truncates to `max_len`, right-pads with `PAD`.

### `CancerGeneDataset`
Reads `DATA_CSV` row by row. Returns `(token_ids_tensor, label_tensor)` per sample.

### `MambaCancerClassifier` — Architecture
```
Input (batch, seq_len)
  → Embedding (vocab_size=7 → d_model)
  → Mamba SSM blocks × n_layer  [RMSNorm → selective scan → residual]
  → Mean pool over seq_len  → (batch, d_model)
  → LayerNorm
  → Linear (d_model → 2)  → logits
  → argmax → 0=Normal  1=Cancer
```

### `metrics(preds, labels)`
Returns `acc`, `prec`, `rec`, `f1` from TP/FP/FN/TN counts.

### `save_ckpt` / `load_ckpt`
Save/restore full training state: model weights + optimizer + scheduler + step number.

In [ ]:
import csv, math, time
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torch.amp import GradScaler, autocast
from transformers import get_cosine_schedule_with_warmup
from mamba_ssm.models.config_mamba import MambaConfig
from mamba_ssm.models.mixer_seq_simple import MixerModel

class DNATokenizer:
    VOCAB = {'[PAD]':0,'[UNK]':1,'A':2,'T':3,'G':4,'C':5,'N':6}
    PAD_ID, vocab_size = 0, 7
    def encode(self, seq, max_len):
        ids = [self.VOCAB.get(c.upper(), 1) for c in seq][:max_len]
        return ids + [0] * (max_len - len(ids))

class CancerGeneDataset(Dataset):
    def __init__(self, path, tok, seq_len):
        self.tok, self.seq_len = tok, seq_len
        self.seqs, self.labels = [], []
        with open(path, newline='') as f:
            for r in csv.DictReader(f):
                self.seqs.append(r['sequence'].strip().upper())
                self.labels.append(int(r['label']))
        c = sum(self.labels)
        print(f'  Loaded {len(self.seqs):,} — Cancer:{c:,}  Normal:{len(self.labels)-c:,}')
    def __len__(self): return len(self.seqs)
    def __getitem__(self, i):
        return (torch.tensor(self.tok.encode(self.seqs[i], self.seq_len), dtype=torch.long),
                torch.tensor(self.labels[i], dtype=torch.long))

class MambaCancerClassifier(nn.Module):
    def __init__(self, cfg, n_cls=2):
        super().__init__()
        self.backbone = MixerModel(
            d_model=cfg.d_model, n_layer=cfg.n_layer,
            d_intermediate=cfg.d_intermediate, vocab_size=cfg.vocab_size,
            ssm_cfg=cfg.ssm_cfg, rms_norm=cfg.rms_norm,
            residual_in_fp32=cfg.residual_in_fp32, fused_add_norm=cfg.fused_add_norm)
        self.norm = nn.LayerNorm(cfg.d_model)
        self.head = nn.Linear(cfg.d_model, n_cls)
    def forward(self, x):
        return self.head(self.norm(self.backbone(x).mean(1)))

def metrics(preds, labels):
    tp=((preds==1)&(labels==1)).sum().item(); fp=((preds==1)&(labels==0)).sum().item()
    fn=((preds==0)&(labels==1)).sum().item(); tn=((preds==0)&(labels==0)).sum().item()
    p=tp/(tp+fp+1e-8); r=tp/(tp+fn+1e-8)
    return {'acc':(tp+tn)/(tp+fp+fn+tn+1e-8),'prec':p,'rec':r,'f1':2*p*r/(p+r+1e-8)}

def save_ckpt(path, step, model, opt, sched, cfg):
    torch.save({'step':step,'cfg':cfg,'model':model.state_dict(),
                'opt':opt.state_dict(),'sched':sched.state_dict()}, path)
    print(f'  [saved] {path}')

def load_ckpt(path, model, opt, sched):
    ck = torch.load(path, map_location=device, weights_only=False)
    model.load_state_dict(ck['model']); opt.load_state_dict(ck['opt'])
    sched.load_state_dict(ck['sched']); return ck['step']

print('Classes ready.')

---
## Cell 7 — Config

> All hyperparameters in one dict. Cell 8 reads these to build everything.

### Model
| Parameter | Default | Description |
|-----------|---------|-------------|
| `d_model` | `256` | Hidden dim: `256`→10M params, `512`→60M, `768`→130M |
| `n_layer` | `8` | Number of Mamba blocks |
| `d_intermediate` | `0` | MLP hidden dim between blocks (`0` = no MLP) |
| `ssm_layer` | `'Mamba1'` | `'Mamba1'` (original) or `'Mamba2'` (SSD) |
| `num_classes` | `2` | Cancer / Normal |

### Data split
| Parameter | Default | Description |
|-----------|---------|-------------|
| `seq_len` | `512` | Tokens per sequence. Must match dataset build. |
| `val_split` | `0.1` | 10% held out for validation during training |
| `test_split` | `0.1` | 10% held out as **unseen test set** (used only in Cell 11) |

> **Split breakdown:** with `val_split=0.1` and `test_split=0.1` the data is divided **80% train / 10% val / 10% test**.

### Training
| Parameter | Default | Description |
|-----------|---------|-------------|
| `batch_size` | `16` | Reduce if OOM. |
| `grad_accum` | `2` | Effective batch = `batch_size × grad_accum` = 32 |
| `lr` | `1e-4` | Peak LR for AdamW |
| `weight_decay` | `0.01` | Applied to weight matrices only |
| `warmup_steps` | `50` | Linear LR ramp before cosine decay |
| `max_steps` | `1000` | Total training steps |
| `clip_grad` | `1.0` | Gradient norm clip threshold |
| `log_every` | `20` | Print metrics every N steps |
| `save_every` | `200` | Validate + save checkpoint every N steps |
| `resume` | `None` | Path to checkpoint to continue training, e.g. `CHECKPOINT_DIR + '/step_200.pt'` |

In [ ]:
CFG = dict(
    # Model
    d_model        = 256,
    n_layer        = 8,
    d_intermediate = 0,
    ssm_layer      = 'Mamba1',
    num_classes    = 2,
    # Data split  (80% train / 10% val / 10% test)
    seq_len        = 512,
    val_split      = 0.1,
    test_split     = 0.1,
    # Training
    batch_size     = 16,
    grad_accum     = 2,
    lr             = 1e-4,
    weight_decay   = 0.01,
    warmup_steps   = 50,
    max_steps      = 50000,
    clip_grad      = 1.0,
    # Logging
    log_every      = 20,
    save_every     = 200,
    # Resume — continues from the last checkpoint (change filename if needed)
    resume         = CHECKPOINT_DIR + '/final.pt',
)
print('Config:'); [print(f'  {k:<16} = {v}') for k, v in CFG.items()]

---
## Cell 8 — Build

> Builds all training objects. Reads paths from Cell 4 and hyperparameters from Cell 7.

### Data split: 80 / 10 / 10

```
full_ds  (100%)
  ├─ tr_ds   (80%)  → tr_loader   used in Cell 9 (training)
  ├─ v_ds    (10%)  → v_loader    used in Cell 9 (validation during training)
  └─ te_ds   (10%)  → te_loader   used in Cell 11 (final evaluation only)
```

> `te_ds` (test set) is **never seen during training**. It is only used once in Cell 11 to report final performance.

### Objects created

| Object | Type | Purpose |
|--------|------|---------|
| `tok` | `DNATokenizer` | Encodes sequences |
| `tr_loader` | `DataLoader` | Training batches |
| `v_loader` | `DataLoader` | Validation batches (during training) |
| `te_loader` | `DataLoader` | Test batches (Cell 11 only) |
| `model` | `MambaCancerClassifier` | The Mamba model |
| `opt` | `AdamW` | Optimizer |
| `sched` | Cosine+Warmup | LR schedule |
| `scaler` | `GradScaler` | FP16 mixed precision |
| `loss_fn` | `CrossEntropyLoss` | Classification loss |

In [ ]:
tok = DNATokenizer()

# ── Three-way split: chromosome-level (no data leakage) ───────────────────────
# Cell 5f writes dataset/splits/{train,val,test}.csv
# Falls back to random 80/10/10 if splits/ not found (for quick experiments).
_split_dir = Path(PIPELINE_DIR) / 'dataset' / 'splits'
_use_chrom_split = (_split_dir / 'train.csv').exists()

if _use_chrom_split:
    print('Using chromosome-level split (no leakage) from dataset/splits/')
    tr_ds = CancerGeneDataset(str(_split_dir / 'train.csv'), tok, CFG['seq_len'])
    v_ds  = CancerGeneDataset(str(_split_dir / 'val.csv'),   tok, CFG['seq_len'])
    te_ds = CancerGeneDataset(str(_split_dir / 'test.csv'),  tok, CFG['seq_len'])
    tr_size, v_size, te_size = len(tr_ds), len(v_ds), len(te_ds)
else:
    print('WARNING: dataset/splits/ not found. Falling back to random split.')
    print('Run Cell 5f to build chromosome-level splits (recommended for paper).')
    full_ds = CancerGeneDataset(DATA_CSV, tok, CFG['seq_len'])
    n       = len(full_ds)
    te_size = max(1, int(n * CFG['test_split']))
    v_size  = max(1, int(n * CFG['val_split']))
    tr_size = n - v_size - te_size
    tr_ds, v_ds, te_ds = torch.utils.data.random_split(
        full_ds, [tr_size, v_size, te_size],
        generator=torch.Generator().manual_seed(42))


tr_loader = DataLoader(tr_ds, CFG['batch_size'], shuffle=True,  drop_last=True,  num_workers=2)
v_loader  = DataLoader(v_ds,  CFG['batch_size'], shuffle=False, drop_last=False, num_workers=2)
te_loader = DataLoader(te_ds, CFG['batch_size'], shuffle=False, drop_last=False, num_workers=2)
print(f'Train:{tr_size:,}  Val:{v_size:,}  Test:{te_size:,}  (total {n:,})')

# ── Model ─────────────────────────────────────────────────────────────────────
mcfg  = MambaConfig(d_model=CFG['d_model'], n_layer=CFG['n_layer'],
                    d_intermediate=CFG['d_intermediate'], vocab_size=tok.vocab_size,
                    ssm_cfg={'layer':CFG['ssm_layer']},
                    rms_norm=True, residual_in_fp32=True, fused_add_norm=True)
model = MambaCancerClassifier(mcfg, CFG['num_classes']).to(device)
print(f'Params: {sum(p.numel() for p in model.parameters())/1e6:.2f}M')

# ── Optimizer ─────────────────────────────────────────────────────────────────
dec, no_dec = [], []
for nm, p in model.named_parameters():
    if p.requires_grad:
        (no_dec if p.ndim<2 or any(x in nm for x in ['bias','norm']) else dec).append(p)
opt   = torch.optim.AdamW([{'params':dec,'weight_decay':CFG['weight_decay']},
                             {'params':no_dec,'weight_decay':0.}],
                            lr=CFG['lr'], betas=(0.9,0.95))
sched = get_cosine_schedule_with_warmup(opt, CFG['warmup_steps'], CFG['max_steps'])
scaler, loss_fn = GradScaler('cuda'), nn.CrossEntropyLoss()

# ── Resume ────────────────────────────────────────────────────────────────────
step = 0
if CFG['resume'] and os.path.exists(CFG['resume']):
    step = load_ckpt(CFG['resume'], model, opt, sched)
    print(f'Resumed from step {step}')
print('Ready.')

---
## Cell 9 — Train

> Runs the training loop and records history for plots in Cell 11.

### Loop summary
```
Each step:
  forward (FP16 autocast) → loss / grad_accum → backward
  every grad_accum steps: unscale → clip → optimizer step → LR step
  every log_every steps:  print train metrics + append to train_history
  every save_every steps: validate → append to val_history → save checkpoint
After max_steps: save final.pt
```

### Log format
```
step   200 | loss 0.6821 | acc 0.612 | f1 0.589 | prec 0.601 | rec 0.578 | 47s
  [VAL step 200] loss 0.6543 | acc 0.638 | f1 0.621
```

### History recorded
- `train_history`: list of `{'step', 'loss', 'acc', 'f1'}` — one entry every `log_every` steps
- `val_history`: list of `{'step', 'loss', 'acc', 'f1'}` — one entry every `save_every` steps

These are used by Cell 11 to plot learning curves.

### Checkpoint files
```
CHECKPOINT_DIR/
  step_200.pt  step_400.pt  ...  final.pt
```
Resume anytime: set `CFG['resume'] = CHECKPOINT_DIR + '/step_N.pt'` in Cell 7, re-run 8 → 9.

> **Expected time on Colab T4:** ~300 sequences, d_model=256, 1000 steps → **5–10 min**.

In [ ]:
train_history, val_history = [], []
model.train(); opt.zero_grad()
rl, ap, al, t0 = 0., [], [], time.time()

for _ in range(99999):
    for ids, lbs in tr_loader:
        if step >= CFG['max_steps']: break
        ids, lbs = ids.to(device), lbs.to(device)
        with autocast('cuda', dtype=torch.float16):
            loss = loss_fn(model(ids), lbs) / CFG['grad_accum']
        scaler.scale(loss).backward()
        if (step+1) % CFG['grad_accum'] == 0:
            scaler.unscale_(opt)
            nn.utils.clip_grad_norm_(model.parameters(), CFG['clip_grad'])
            scaler.step(opt); scaler.update(); sched.step(); opt.zero_grad()
        rl += loss.item()*CFG['grad_accum']
        ap.append(model(ids).argmax(-1).detach().cpu()); al.append(lbs.cpu())
        step += 1

        if step % CFG['log_every'] == 0:
            m = metrics(torch.cat(ap), torch.cat(al))
            avg_loss = rl / CFG['log_every']
            print(f"step {step:>5} | loss {avg_loss:.4f} | "
                  f"acc {m['acc']:.3f} | f1 {m['f1']:.3f} | "
                  f"prec {m['prec']:.3f} | rec {m['rec']:.3f} | "
                  f"{time.time()-t0:.0f}s")
            train_history.append({'step':step,'loss':avg_loss,'acc':m['acc'],'f1':m['f1']})
            rl, ap, al, t0 = 0., [], [], time.time()

        if step % CFG['save_every'] == 0:
            model.eval()
            vp, vl, vl_ = [], [], 0.
            with torch.no_grad():
                for vi, vb in v_loader:
                    vi, vb = vi.to(device), vb.to(device)
                    vlog = model(vi); vl_ += loss_fn(vlog, vb).item()
                    vp.append(vlog.argmax(-1).cpu()); vl.append(vb.cpu())
            vm = metrics(torch.cat(vp), torch.cat(vl))
            avg_vloss = vl_/len(v_loader)
            print(f"\n  [VAL step {step}] loss {avg_vloss:.4f} | "
                  f"acc {vm['acc']:.3f} | f1 {vm['f1']:.3f}\n")
            val_history.append({'step':step,'loss':avg_vloss,'acc':vm['acc'],'f1':vm['f1']})
            save_ckpt(f"{CHECKPOINT_DIR}/step_{step}.pt", step, model, opt, sched, mcfg)
            model.train()

    if step >= CFG['max_steps']: break

save_ckpt(f'{CHECKPOINT_DIR}/final.pt', step, model, opt, sched, mcfg)
print('Training complete.')

---
## Cell 8b -- Dry-Run (100 steps, sanity check)

> Run before full training to verify the pipeline loads correctly.
> Expected: loss decreasing, AUC printed at step 50 and 100, exit code 0.

In [ ]:
# -- Cell 8b -- Dry-Run (100 steps, sanity check)
# Prerequisite: Cell 5f must have been run (splits must exist in dataset/splits/)
import subprocess, os, sys
from pathlib import Path

SPLITS = str(Path(PIPELINE_DIR) / "dataset" / "splits")
TRAIN  = str(Path(PIPELINE_DIR) / "train.py")

if not (Path(SPLITS) / "train.csv").exists():
    print("ERROR: splits not found. Run Cell 5f first.")
else:
    cmd = [sys.executable, TRAIN,
           "--splits_dir", SPLITS,
           "--max_steps",  "100",
           "--log_every",  "20",
           "--save_every", "50",
           "--batch_size", "16",
           "--save_dir",   str(Path(PIPELINE_DIR) / "checkpoints_dryrun")]

    print("Dry-run command:", " ".join(cmd))
    print("=" * 60)

    proc = subprocess.Popen(cmd, stdout=subprocess.PIPE,
                            stderr=subprocess.STDOUT, text=True,
                            bufsize=1,
                            env={**os.environ, "PYTHONUNBUFFERED": "1"})
    for line in proc.stdout:
        print(line, end="", flush=True)
    proc.wait()
    print(f"
===== exit code: {proc.returncode} =====")


---
## Cell 10 — Inference

> Test the trained model on any raw DNA sequence.

### Loading a checkpoint (optional)
Uncomment the first line to reload a saved checkpoint:
```python
load_ckpt(CHECKPOINT_DIR + '/final.pt', model, opt, sched)
```

### Adding your own sequences
Edit `test_seqs` — paste any DNA string (A/T/G/C/N).  
Sequences > `seq_len` are truncated; shorter ones are padded.

### Reading the output
```
ATCGATCGNNGATCGATCG...
→ CANCER  (Cancer=0.823  Normal=0.177)
```
The two probabilities always sum to 1.0.

> The default sequences are synthetic examples only — use real genomic sequences for meaningful results.

In [ ]:
# Optional: load a specific checkpoint
# load_ckpt(CHECKPOINT_DIR + '/final.pt', model, opt, sched)

model.eval()
test_seqs = [
    'ATCGATCGNNGATCGATCGATCGATCGATCGATCGATCGATCGATCGATCG',
    'GCTAGCTAGCTAGCTAGCTAGCTAGCTAGCTAGCTAGCTAGCTAGCTAGCT',
    'AAAATTTTCCCCGGGGAAAATTTTCCCCGGGGAAAATTTTCCCCGGGGAAAA',
]
print('Results'); print('='*60)
for seq in test_seqs:
    ids = torch.tensor([tok.encode(seq, CFG['seq_len'])], dtype=torch.long).to(device)
    with torch.no_grad():
        pr = torch.softmax(model(ids), -1)[0]
    label = 'CANCER' if pr.argmax().item()==1 else 'NORMAL'
    print(f'  {seq[:35]}...')
    print(f'  → {label}  (Cancer={pr[1]:.3f}  Normal={pr[0]:.3f})')
    print('-'*60)

---
## Cell 11 — Evaluation

> Runs the **held-out test set** through the trained model and produces three plots.
> This is the first and only time `te_loader` (the 10% test split) is used.

### What this cell produces

| Output | Description |
|--------|-------------|
| **Test metrics table** | Accuracy, Precision, Recall, F1, AUC on the unseen test set |
| **Plot 1 — Learning curves** | Training loss + val loss / F1 over steps (from `train_history` / `val_history`) |
| **Plot 2 — ROC curve** | True positive rate vs false positive rate, with AUC score |
| **Plot 3 — Confusion matrix** | TP / FP / FN / TN counts on the test set |

### Why test set matters
The model never saw `te_ds` during training or validation.  
Reporting metrics on the test set is the honest estimate of real-world performance.  
Val metrics (from Cell 9) are optimistic — the model was tuned while those were visible.

### AUC (Area Under ROC Curve)
- `AUC = 1.0` → perfect classifier  
- `AUC = 0.5` → random classifier  
- `AUC > 0.80` → generally considered good for cancer detection

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import roc_curve, auc, confusion_matrix, ConfusionMatrixDisplay

# ── Auto-select best checkpoint by lowest val loss ────────────────────────────
if val_history:
    best = min(val_history, key=lambda h: h['loss'])
    best_ckpt = f"{CHECKPOINT_DIR}/step_{best['step']}.pt"
    print(f"Best checkpoint: step_{best['step']}.pt  "
          f"(val loss {best['loss']:.4f}, val acc {best['acc']:.3f})")
    load_ckpt(best_ckpt, model, opt, sched)
else:
    # val_history empty (runtime was restarted) — set manually if needed
    best_ckpt = CHECKPOINT_DIR + '/step_37600.pt'
    print(f"val_history empty — loading {best_ckpt}")
    load_ckpt(best_ckpt, model, opt, sched)

# ── Run test set ──────────────────────────────────────────────────────────────
model.eval()
all_preds, all_labels, all_probs = [], [], []
with torch.no_grad():
    for ids, lbs in te_loader:
        ids = ids.to(device)
        logits = model(ids)
        probs  = torch.softmax(logits, -1)[:, 1].cpu()
        preds  = logits.argmax(-1).cpu()
        all_probs.append(probs); all_preds.append(preds); all_labels.append(lbs)

all_preds  = torch.cat(all_preds)
all_labels = torch.cat(all_labels)
all_probs  = torch.cat(all_probs).numpy()

# ── Test metrics ──────────────────────────────────────────────────────────────
tm   = metrics(all_preds, all_labels)
fpr, tpr, _ = roc_curve(all_labels.numpy(), all_probs)
roc_auc     = auc(fpr, tpr)

print('\n' + '═'*50)
print('  TEST SET RESULTS (unseen during training)')
print('═'*50)
print(f"  Accuracy  : {tm['acc']:.4f}")
print(f"  Precision : {tm['prec']:.4f}")
print(f"  Recall    : {tm['rec']:.4f}")
print(f"  F1 Score  : {tm['f1']:.4f}")
print(f"  AUC       : {roc_auc:.4f}")
print('═'*50)

# ── Plot 1: Learning curves ───────────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

if train_history:
    tr_steps = [h['step'] for h in train_history]
    tr_loss  = [h['loss'] for h in train_history]
    tr_f1    = [h['f1']   for h in train_history]
    axes[0].plot(tr_steps, tr_loss,  label='Train loss',  color='steelblue')
    axes[0].plot(tr_steps, tr_f1,    label='Train F1',    color='steelblue', linestyle='--')
if val_history:
    v_steps  = [h['step'] for h in val_history]
    v_loss   = [h['loss'] for h in val_history]
    v_f1     = [h['f1']   for h in val_history]
    axes[0].plot(v_steps, v_loss,    label='Val loss',    color='darkorange')
    axes[0].plot(v_steps, v_f1,      label='Val F1',      color='darkorange',  linestyle='--')
axes[0].set_title('Learning Curves'); axes[0].set_xlabel('Step')
axes[0].legend(); axes[0].grid(True, alpha=0.3)

# ── Plot 2: ROC curve ─────────────────────────────────────────────────────────
axes[1].plot(fpr, tpr, color='steelblue', lw=2, label=f'Mamba (AUC = {roc_auc:.3f})')
axes[1].plot([0,1],[0,1],'k--', lw=1, label='Random (AUC = 0.500)')
axes[1].set_xlabel('False Positive Rate'); axes[1].set_ylabel('True Positive Rate')
axes[1].set_title('ROC Curve'); axes[1].legend(); axes[1].grid(True, alpha=0.3)

# ── Plot 3: Confusion matrix ──────────────────────────────────────────────────
cm = confusion_matrix(all_labels.numpy(), all_preds.numpy())
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=['Normal','Cancer'])
disp.plot(ax=axes[2], colorbar=False, cmap='Blues')
axes[2].set_title('Confusion Matrix (Test Set)')

plt.tight_layout()
plt.savefig(f'{CHECKPOINT_DIR}/evaluation_plots.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'  Plots saved to {CHECKPOINT_DIR}/evaluation_plots.png')

# Store for Cell 13
MAMBA_RESULTS = {'acc':tm['acc'],'prec':tm['prec'],'rec':tm['rec'],'f1':tm['f1'],'auc':roc_auc}

---
## Cell 12 — Baseline

> Trains a **Logistic Regression** classifier on the same train/test split using **k-mer frequency features**.
> Provides a fair comparison to show whether Mamba adds value over a classical method.

### Why this baseline matters
Without a baseline, you can't claim Mamba is useful for this task.  
If the logistic regression achieves similar F1 and AUC, the sequence model adds no value.  
If Mamba scores higher, the result is meaningful.

### What are k-mer features?
A **k-mer** is a substring of length `k`. For `k=3` (trinucleotides) there are 4³ = 64 possible 3-mers (AAA, AAT, …, TTT).  
Each sequence is converted to a 64-dimensional vector of 3-mer frequencies — a standard representation in computational biology.

```
sequence: 'ATCGATCG...'  →  count all overlapping 3-mers  →  64-dim vector
```

### Why Logistic Regression?
- Fast to train (seconds vs minutes for Mamba)
- Interpretable (feature weights = most discriminative k-mers)
- Standard baseline in genomics classification papers
- Uses **the same train/test split** as Cell 8 for a fair comparison

### What k-mer size is used?
`KMER_SIZE = 3` (trinucleotides, 64 features). You can change this — larger k captures more context but increases feature space: `k=4` → 256 features, `k=5` → 1024 features.

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_curve, auc
from itertools import product

KMER_SIZE = 3   # change to 4 or 5 for richer features

# ── Build k-mer vocabulary ────────────────────────────────────────────────────
BASES = ['A', 'T', 'G', 'C']
KMERS = [''.join(k) for k in product(BASES, repeat=KMER_SIZE)]
KMER_IDX = {k: i for i, k in enumerate(KMERS)}

def seq_to_kmer_features(seq):
    vec = np.zeros(len(KMERS), dtype=np.float32)
    seq = seq.upper().replace('N', 'A')   # replace ambiguous bases
    total = max(1, len(seq) - KMER_SIZE + 1)
    for i in range(total):
        kmer = seq[i:i+KMER_SIZE]
        if kmer in KMER_IDX:
            vec[KMER_IDX[kmer]] += 1
    return vec / total   # normalize to frequencies

# ── Extract features from train and test subsets ──────────────────────────────
print(f'Building {KMER_SIZE}-mer features ({len(KMERS)} dims) ...')

def extract_features(subset):
    feats, lbls = [], []
    for idx in subset.indices:
        seq   = full_ds.seqs[idx]
        label = full_ds.labels[idx]
        feats.append(seq_to_kmer_features(seq))
        lbls.append(label)
    return np.array(feats), np.array(lbls)

X_train, y_train = extract_features(tr_ds)
X_test,  y_test  = extract_features(te_ds)
print(f'Train: {X_train.shape}  Test: {X_test.shape}')

# ── Train Logistic Regression ─────────────────────────────────────────────────
print('Training Logistic Regression ...')
lr_clf = LogisticRegression(max_iter=1000, C=1.0, random_state=42)
lr_clf.fit(X_train, y_train)

# ── Evaluate on test set ──────────────────────────────────────────────────────
lr_preds = lr_clf.predict(X_test)
lr_probs = lr_clf.predict_proba(X_test)[:, 1]

lr_preds_t = torch.tensor(lr_preds)
lr_labels_t = torch.tensor(y_test)
bm = metrics(lr_preds_t, lr_labels_t)

fpr_b, tpr_b, _ = roc_curve(y_test, lr_probs)
auc_b = auc(fpr_b, tpr_b)

print('\n' + '═'*50)
print(f'  BASELINE: Logistic Regression ({KMER_SIZE}-mer features)')
print('═'*50)
print(f"  Accuracy  : {bm['acc']:.4f}")
print(f"  Precision : {bm['prec']:.4f}")
print(f"  Recall    : {bm['rec']:.4f}")
print(f"  F1 Score  : {bm['f1']:.4f}")
print(f"  AUC       : {auc_b:.4f}")
print('═'*50)

# ── Top discriminative k-mers ─────────────────────────────────────────────────
coef = lr_clf.coef_[0]
top_cancer = [KMERS[i] for i in np.argsort(coef)[-5:][::-1]]
top_normal = [KMERS[i] for i in np.argsort(coef)[:5]]
print(f'\n  Top cancer-associated {KMER_SIZE}-mers : {top_cancer}')
print(f'  Top normal-associated {KMER_SIZE}-mers  : {top_normal}')

# Store for Cell 13
BASELINE_RESULTS = {'acc':bm['acc'],'prec':bm['prec'],'rec':bm['rec'],'f1':bm['f1'],'auc':auc_b,
                    'fpr':fpr_b,'tpr':tpr_b}

---
## Cell 13 — Results Summary

> Side-by-side comparison of Mamba vs. Logistic Regression baseline on the same test set.
> Also overlays both ROC curves on one plot for visual comparison.

### How to interpret the results

| Scenario | Interpretation |
|----------|----------------|
| Mamba F1 and AUC clearly higher | Mamba's sequence scanning captures patterns the k-mer model misses |
| Mamba ≈ Baseline | The signal is already captured by local k-mer statistics; Mamba adds no value |
| Mamba lower | Dataset too small, or training needs more steps / tuning |

### What to report in a paper
- Report **test set** metrics (not val metrics)
- Report AUC alongside F1 — AUC is threshold-independent
- Report the confusion matrix (TP/FP/FN/TN) to show clinical relevance
- State the train/val/test split clearly (80/10/10 here)

In [ ]:
import matplotlib.pyplot as plt

# ── Summary table ─────────────────────────────────────────────────────────────
print('\n' + '═'*62)
print(f"  {'Metric':<14}  {'Mamba (ours)':>14}  {'LR Baseline':>14}  {'Δ':>8}")
print('─'*62)
for key in ['acc','prec','rec','f1','auc']:
    m_val = MAMBA_RESULTS[key]
    b_val = BASELINE_RESULTS[key]
    delta = m_val - b_val
    sign  = '+' if delta >= 0 else ''
    label = key.upper() if key != 'acc' else 'Accuracy'
    print(f"  {label:<14}  {m_val:>14.4f}  {b_val:>14.4f}  {sign}{delta:>7.4f}")
print('═'*62)
print(f"  Split: 80% train / 10% val / 10% test")
print(f"  Mamba: d_model={CFG['d_model']}  n_layer={CFG['n_layer']}  "
      f"seq_len={CFG['seq_len']}  steps={CFG['max_steps']}")
print(f"  Baseline: Logistic Regression on {KMER_SIZE}-mer frequencies ({len(KMERS)} features)")
print('═'*62)

# ── Overlaid ROC curves ───────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(7, 6))

# Mamba ROC (recompute from MAMBA_RESULTS — we need fpr/tpr arrays)
# Re-run inference to get probabilities if MAMBA_RESULTS doesn't have them
model.eval()
all_probs_m, all_labels_m = [], []
with torch.no_grad():
    for ids, lbs in te_loader:
        probs = torch.softmax(model(ids.to(device)), -1)[:, 1].cpu()
        all_probs_m.append(probs); all_labels_m.append(lbs)
all_probs_m = torch.cat(all_probs_m).numpy()
all_labels_m = torch.cat(all_labels_m).numpy()
fpr_m, tpr_m, _ = roc_curve(all_labels_m, all_probs_m)
auc_m = auc(fpr_m, tpr_m)

ax.plot(fpr_m, tpr_m, color='steelblue', lw=2,
        label=f"Mamba  (AUC = {auc_m:.3f})")
ax.plot(BASELINE_RESULTS['fpr'], BASELINE_RESULTS['tpr'],
        color='darkorange', lw=2,
        label=f"LR Baseline (AUC = {BASELINE_RESULTS['auc']:.3f})")
ax.plot([0,1],[0,1],'k--', lw=1, label='Random (AUC = 0.500)')
ax.set_xlabel('False Positive Rate', fontsize=12)
ax.set_ylabel('True Positive Rate', fontsize=12)
ax.set_title('ROC Curve — Mamba vs Baseline', fontsize=13)
ax.legend(fontsize=11); ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(f'{CHECKPOINT_DIR}/roc_comparison.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'  Saved to {CHECKPOINT_DIR}/roc_comparison.png')

---
## Session End — Save Results to Drive

> Run this cell **before closing Colab** to copy all generated files back to Drive.
> Your Drive folder is always the canonical project — this keeps it in sync.

In [ ]:
# ── Session End -- Save results back to Drive (skip if unchanged) ────────
import shutil
from pathlib import Path

LOCAL = Path(PIPELINE_DIR)
DRIVE = Path(DRIVE_DIR)

SAVE_BACK = [
    # Sequences
    'sequences/normals_windows_matched.jsonl.gz',
    'sequences/normals_windows.jsonl.gz',
    # Datasets
    'dataset/cancer_genes_matched.csv',
    'dataset/cancer_genes.csv',
    # Diagnostics
    'diagnostics/gc_stats.json',
    'diagnostics/chrom_stats.json',
    'diagnostics/dinuc_stats.json',
    'diagnostics/summary.md',
    # FASTA index (small; avoids re-build next session)
    'GRCh38_no_alt.fna.fai',
    # GENCODE GTF (large; save so next session skips download)
    'diagnostics/tmp_matching/gencode.v38.annotation.gtf.gz',
    'diagnostics/tmp_matching/gencode.v38.annotation.bed',
]

# Add all checkpoint .pt files found in workspace
ckpt_dir = LOCAL / 'checkpoints'
if ckpt_dir.exists():
    for cp in ckpt_dir.glob('*.pt'):
        SAVE_BACK.append(f'checkpoints/{cp.name}')

print('Saving results back to Drive ...')
n_saved = n_skipped = n_missing = 0
for rel in SAVE_BACK:
    src = LOCAL / rel
    if not src.exists() or src.is_symlink():
        n_missing += 1
        continue
    dst = DRIVE / rel
    dst.parent.mkdir(parents=True, exist_ok=True)
    if dst.exists() and dst.stat().st_size == src.stat().st_size:
        print(f'  SKIP   {src.stat().st_size/1e6:7.1f} MB  {rel}')
        n_skipped += 1
    else:
        shutil.copy2(src, dst)
        print(f'  SAVED  {src.stat().st_size/1e6:7.1f} MB  {rel}')
        n_saved += 1

print(f'\nDone: {n_saved} saved, {n_skipped} unchanged, {n_missing} not yet generated.')
